# Bootstrap Classifier Evaluation - Starter Notebook
This notebook demonstrates bootstrapping for estimating model performance and confidence intervals.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="wine_class")

rng = np.random.default_rng(42)
n_bootstraps = 100
bootstrap_scores = []

X.shape

In [ ]:
for i in range(n_bootstraps):
    indices = rng.choice(X.shape[0], size=X.shape[0], replace=True)
    oob_indices = np.setdiff1d(np.arange(X.shape[0]), indices)
    
    X_boot = X.iloc[indices]
    y_boot = y.iloc[indices]
    X_oob = X.iloc[oob_indices]
    y_oob = y.iloc[oob_indices]
    
    if len(y_oob) == 0:
        continue
    
    model = RandomForestClassifier(n_estimators=50, random_state=i)
    model.fit(X_boot, y_boot)
    score = accuracy_score(y_oob, model.predict(X_oob))
    bootstrap_scores.append(score)

bootstrap_scores = np.array(bootstrap_scores)
print(f"Mean Bootstrap Score: {bootstrap_scores.mean():.3f}")
print(f"Std Dev: {bootstrap_scores.std():.3f}")
print(f"95% CI: [{np.percentile(bootstrap_scores, 2.5):.3f}, {np.percentile(bootstrap_scores, 97.5):.3f}]")

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(bootstrap_scores, bins=20, edgecolor="black", alpha=0.7)
plt.axvline(bootstrap_scores.mean(), color="red", linestyle="--", label=f"Mean: {bootstrap_scores.mean():.3f}")
plt.title("Bootstrap Score Distribution")
plt.xlabel("Accuracy")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()